# TRIBE v2 + SurfaceFsaverageDecoderBackend

This notebook runs TRIBE v2 only to produce fsaverage5 `.npy` maps, then decodes them with cached fsaverage5 Neurosynth reference maps. `build-references` uses precomputed Neurosynth association maps and does not fit a full NiMARE decoder by default.

In [ ]:
# -*- coding: utf-8 -*-
import os
import subprocess
import torch

os.environ.setdefault("PYTHONUTF8", "1")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)
if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. Select a Colab GPU kernel.")
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print(f"GPU: {gpu_name}, VRAM: {vram_gb:.1f} GB")


In [ ]:
# -*- coding: utf-8 -*-
import base64
from pathlib import Path

PROJECT_DIR = Path("/content/neuromedia")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TRIBE_B64 = "..."
SURFACE_DECODER_B64 = "..."
REQUIREMENTS_B64 = "bnVtcHkKcGFuZGFzCnNjaXB5Cm5pYmFiZWwKbmlsZWFybgpuaW1hcmUKdHJpYmV2MltwbG90dGluZ10gQCBnaXQraHR0cHM6Ly9naXRodWIuY29tL2ZhY2Vib29rcmVzZWFyY2gvdHJpYmV2Mi5naXQK"

(PROJECT_DIR / "tribe_nimare_interpreter.py").write_text(
    base64.b64decode(TRIBE_B64).decode("utf-8"), encoding="utf-8"
)
(PROJECT_DIR / "marketing_surface_decoder.py").write_text(
    base64.b64decode(SURFACE_DECODER_B64).decode("utf-8"), encoding="utf-8"
)
(PROJECT_DIR / "requirements.txt").write_text(
    base64.b64decode(REQUIREMENTS_B64).decode("utf-8"), encoding="utf-8"
)

%cd /content/neuromedia
!ls -la


In [ ]:
# -*- coding: utf-8 -*-
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "/content/neuromedia/requirements.txt"], check=True)


In [ ]:
# -*- coding: utf-8 -*-
import os
from huggingface_hub import login, whoami

# Paste your Hugging Face read token here before running this cell.
HF_TOKEN = ""
token = HF_TOKEN.strip()
if not token:
    raise RuntimeError("Set HF_TOKEN in this cell before running it.")
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
user = whoami()
print(f"Logged in to Hugging Face as: {user.get('name', 'unknown')}")


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/neuromedia")
INPUT_DIR = DRIVE_DIR / "input"
OUTPUT_ROOT = DRIVE_DIR / "outputs" / "surface_decoder"
TRIBE_OUTPUT_ROOT = DRIVE_DIR / "outputs" / "tribe_fsaverage5"
TRIBE_CACHE_DIR = DRIVE_DIR / "cache" / "tribev2"
NIMARE_DATA_DIR = DRIVE_DIR / "cache" / "nimare"
REFERENCES_DIR = DRIVE_DIR / "cache" / "surface_references" / "marketing_v1"
SUPPORTED_EXTENSIONS = {".txt", ".wav", ".mp3", ".flac", ".ogg", ".mp4", ".avi", ".mkv", ".mov", ".webm"}
INPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in (OUTPUT_ROOT, TRIBE_OUTPUT_ROOT, TRIBE_CACHE_DIR, NIMARE_DATA_DIR, REFERENCES_DIR):
    path.mkdir(parents=True, exist_ok=True)
INPUT_FILES = sorted(path for path in INPUT_DIR.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS)
if not INPUT_FILES:
    raise FileNotFoundError(f"No supported inputs found in {INPUT_DIR}")
for input_file in INPUT_FILES:
    print(input_file)


In [ ]:
# -*- coding: utf-8 -*-
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONUTF8"] = "1"

def output_dir_for_input(input_file):
    return TRIBE_OUTPUT_ROOT.joinpath(*input_file.relative_to(INPUT_DIR).with_suffix("").parts)

for index, input_file in enumerate(INPUT_FILES, start=1):
    output_dir = output_dir_for_input(input_file)
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "/content/neuromedia/tribe_nimare_interpreter.py", str(input_file),
        "--output-dir", str(output_dir),
        "--cache-dir", str(TRIBE_CACHE_DIR),
        "--device", "cuda",
        "--tribe-only",
    ]
    if (output_dir / "tribe_activity_fsaverage5.npy").is_file():
        cmd.append("--reuse-tribe")
    print(f"
[{index}/{len(INPUT_FILES)}] TRIBE-only: {input_file}")
    subprocess.run(cmd, check=True, env=env)


In [ ]:
# -*- coding: utf-8 -*-
import subprocess
import sys

cmd = [
    sys.executable, "/content/neuromedia/marketing_surface_decoder.py", "build-references",
    "--references-dir", str(REFERENCES_DIR),
    "--reference-source", "neurosynth_precomputed",
    "--neurosynth-cache-dir", str(DRIVE_DIR / "cache" / "neurosynth_precomputed"),
    "--n-cores", "1",
]
subprocess.run(cmd, check=True)


In [ ]:
# -*- coding: utf-8 -*-
import subprocess
import sys

activity_files = sorted(TRIBE_OUTPUT_ROOT.rglob("tribe_activity_fsaverage5.npy"))
if not activity_files:
    raise FileNotFoundError(f"No TRIBE activity files found under {TRIBE_OUTPUT_ROOT}")
for activity_file in activity_files:
    rel = activity_file.parent.relative_to(TRIBE_OUTPUT_ROOT)
    out_dir = OUTPUT_ROOT.joinpath(*rel.parts)
    cmd = [
        sys.executable, "/content/neuromedia/marketing_surface_decoder.py", "decode", str(activity_file),
        "--references-dir", str(REFERENCES_DIR),
        "--output-dir", str(out_dir),
        "--hemisphere-order", "left_then_right",
    ]
    print(f"
Surface decode: {activity_file}")
    subprocess.run(cmd, check=True)
